# 04 - Normalizar imagenes con Rasterio

Cuarta etapa del flujo Geosupport. Toma las imagenes cargadas al datastore, busca su footprint en el indice de vuelos y genera una version normalizada/cortada con `rasterio.mask`.

Por defecto no reemplaza originales. Primero revisar los outputs; luego activar `REPLACE_ORIGINALS = True` si corresponde.

In [ ]:
from datetime import datetime
from pathlib import Path
import csv
import shutil
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'core').exists():
    for candidate in [Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / 'core').exists():
            PROJECT_ROOT = candidate
            break

if not (PROJECT_ROOT / 'core').exists():
    raise FileNotFoundError('No se encontro el folder core. Ejecuta el notebook desde la raiz del proyecto o desde flujo_geosupport_etapas.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FLOW_DIR = PROJECT_ROOT / 'flujo_geosupport_etapas'

import arcpy
arcpy.env.overwriteOutput = True

print('Proyecto raiz:', PROJECT_ROOT)
print('Folder flujo:', FLOW_DIR)

## Parametros

In [ ]:
LOAD_RESULTS_CSV = FLOW_DIR / 'outputs' / 'etapa_02_carga_datastore_mosaico' / '02_load_results.csv'
LOAD_INPUT_ATTRIBUTES_CSV = FLOW_DIR / 'outputs' / 'etapa_02_carga_datastore_mosaico' / '01_load_input_with_attributes.csv'

PATH_FC_FOOTPRINTS = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO"
FOOTPRINT_NAME_FIELD = 'Name'

# Seguridad operacional.
PROCESS_ONLY_SUCCESSFUL_LOADS = True
LIMIT_ROWS = None
REPLACE_ORIGINALS = False
CREATE_BACKUP_BEFORE_REPLACE = False
BUILD_PYRAMIDS_AFTER_REPLACE = False

# Mantener False evita duplicar .bak en datastore. Activar solo si se requiere respaldo local del original.
BACKUP_SUFFIX = '.bak_original_before_rasterio'

OUTPUT_DIR = FLOW_DIR / 'outputs' / 'etapa_04_normalizar_imagenes_rasterio'
NORMALIZED_DIR = OUTPUT_DIR / 'normalized_tif'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)
run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

print('CSV resultados etapa 2:', LOAD_RESULTS_CSV)
print('CSV input atributos etapa 2:', LOAD_INPUT_ATTRIBUTES_CSV)
print('Feature footprints:', PATH_FC_FOOTPRINTS)
print('Salida normalizados:', NORMALIZED_DIR)
print('REPLACE_ORIGINALS:', REPLACE_ORIGINALS)
print('CREATE_BACKUP_BEFORE_REPLACE:', CREATE_BACKUP_BEFORE_REPLACE)

## 1. Leer imagenes a normalizar

In [ ]:
def read_stage_2_rows():
    if LOAD_RESULTS_CSV.exists():
        df = pd.read_csv(LOAD_RESULTS_CSV)
        if 'destination_path' not in df.columns and LOAD_INPUT_ATTRIBUTES_CSV.exists():
            attrs = pd.read_csv(LOAD_INPUT_ATTRIBUTES_CSV)
            df = df.merge(attrs[['Name', 'destination_path']], on='Name', how='left')
        if PROCESS_ONLY_SUCCESSFUL_LOADS and 'overall_status' in df.columns:
            ok_mask = df['overall_status'].astype(str).str.lower().isin(['ok'])
            if 'mosaic_add_status' in df.columns:
                ok_mask = ok_mask | df['mosaic_add_status'].astype(str).str.lower().isin(['added', 'already_exists'])
            df = df[ok_mask].copy()
        return df

    if LOAD_INPUT_ATTRIBUTES_CSV.exists():
        return pd.read_csv(LOAD_INPUT_ATTRIBUTES_CSV)

    raise FileNotFoundError('No existe salida de etapa 2 para normalizar.')


work_df = read_stage_2_rows()
if 'Name' not in work_df.columns:
    raise ValueError('La entrada debe contener columna Name')
if 'destination_path' not in work_df.columns:
    if 'Path_Destino' in work_df.columns:
        work_df['destination_path'] = work_df['Path_Destino']
    else:
        raise ValueError('La entrada debe contener destination_path o Path_Destino')

work_df = work_df[work_df['Name'].notna() & work_df['destination_path'].notna()].copy()
if LIMIT_ROWS is not None:
    work_df = work_df.head(int(LIMIT_ROWS)).copy()

print(f'Imagenes a normalizar: {len(work_df)}')
display(work_df[['Name', 'destination_path']].head(30))

## 2. Cargar footprints desde el indice

In [ ]:
def load_footprints_index(feature_class, name_field):
    fields = {field.name.lower(): field.name for field in arcpy.ListFields(feature_class)}
    if name_field.lower() not in fields:
        raise ValueError(f'No existe el campo {name_field} en {feature_class}')
    resolved_name_field = fields[name_field.lower()]

    footprints = {}
    duplicates = []
    with arcpy.da.SearchCursor(feature_class, [resolved_name_field, 'SHAPE@']) as cursor:
        for name, geometry in cursor:
            if not name or geometry is None:
                continue
            key = str(name)
            if key in footprints:
                duplicates.append(key)
            footprints[key] = geometry

    if duplicates:
        print('Advertencia: nombres duplicados en footprints:', sorted(set(duplicates))[:20])
    return footprints


footprints_index = load_footprints_index(PATH_FC_FOOTPRINTS, FOOTPRINT_NAME_FIELD)
print(f'Footprints cargados: {len(footprints_index)}')

## 3. Normalizar con Rasterio

El recorte usa la geometria del footprint. Si el raster es RGB se conserva RGB y se agrega alpha para transparencia limpia fuera del poligono. Si es colormap/1 banda, se convierte a RGB cuando existe colormap.

In [ ]:
def arcpy_geometry_to_geojson_dict(geometry, target_spatial_reference):
    import json
    geom = geometry
    if target_spatial_reference and geometry.spatialReference and geometry.spatialReference.factoryCode != target_spatial_reference.factoryCode:
        geom = geometry.projectAs(target_spatial_reference)
    return json.loads(geom.JSON)


def output_path_for_raster(source_path, name):
    source_path = Path(source_path)
    safe_name = f'{name}.tif' if not str(name).lower().endswith('.tif') else str(name)
    return NORMALIZED_DIR / source_path.parent.name / safe_name


def normalize_with_rasterio(source_path, footprint_geometry, output_path):
    import numpy as np
    import rasterio
    from rasterio.mask import mask

    source_path = Path(source_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(source_path) as src:
        raster_sr_code = None
        if src.crs and src.crs.to_epsg():
            raster_sr_code = src.crs.to_epsg()
        raster_sr = arcpy.SpatialReference(raster_sr_code) if raster_sr_code else None
        footprint_geojson = arcpy_geometry_to_geojson_dict(footprint_geometry, raster_sr)

        masked_data, out_transform = mask(src, [footprint_geojson], crop=True, filled=False)
        profile = src.profile.copy()
        profile.update(
            driver='GTiff',
            height=masked_data.shape[1],
            width=masked_data.shape[2],
            transform=out_transform,
            tiled=True,
            compress='deflate',
            photometric='RGB',
        )

        if src.count >= 3:
            rgb = masked_data[:3].filled(0)
        elif src.count == 1:
            values = masked_data[0].filled(0)
            colormap = None
            try:
                colormap = src.colormap(1)
            except Exception:
                colormap = None
            if colormap:
                rgb = np.zeros((3, values.shape[0], values.shape[1]), dtype='uint8')
                for pixel_value, color in colormap.items():
                    color_mask = values == pixel_value
                    if color_mask.any():
                        rgb[0][color_mask] = color[0]
                        rgb[1][color_mask] = color[1]
                        rgb[2][color_mask] = color[2]
            else:
                rgb = np.repeat(values[np.newaxis, :, :], 3, axis=0)
        else:
            raise ValueError(f'Raster con cantidad de bandas no soportada: {src.count}')

        alpha = (~masked_data.mask.all(axis=0)).astype('uint8') * 255
        profile.update(count=4, dtype=rgb.dtype, nodata=None)

        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(rgb.astype(profile['dtype'], copy=False))
            dst.write(alpha, 4)
            dst.colorinterp = (
                rasterio.enums.ColorInterp.red,
                rasterio.enums.ColorInterp.green,
                rasterio.enums.ColorInterp.blue,
                rasterio.enums.ColorInterp.alpha,
            )

    return output_path


def replace_original(source_path, normalized_path):
    source_path = Path(source_path)
    normalized_path = Path(normalized_path)
    backup_path = source_path.with_suffix(source_path.suffix + BACKUP_SUFFIX)
    if CREATE_BACKUP_BEFORE_REPLACE and not backup_path.exists():
        shutil.copy2(source_path, backup_path)
    shutil.copy2(normalized_path, source_path)
    if BUILD_PYRAMIDS_AFTER_REPLACE:
        arcpy.management.BuildPyramidsandStatistics(str(source_path))
    return backup_path if CREATE_BACKUP_BEFORE_REPLACE else None


In [ ]:
results = []
for index, row in work_df.iterrows():
    name = str(row['Name'])
    source_path = Path(row['destination_path'])
    output_path = output_path_for_raster(source_path, name)
    item = {
        'Name': name,
        'source_path': str(source_path),
        'normalized_path': str(output_path),
        'replace_original': REPLACE_ORIGINALS,
        'backup_path': '',
        'status': 'pending',
        'error': '',
    }

    try:
        if not source_path.exists():
            raise FileNotFoundError(f'No existe raster origen: {source_path}')
        footprint_geometry = footprints_index.get(name)
        if footprint_geometry is None:
            raise ValueError(f'No existe footprint para Name={name}')

        normalize_with_rasterio(source_path, footprint_geometry, output_path)
        if REPLACE_ORIGINALS:
            backup_path = replace_original(source_path, output_path)
            item['backup_path'] = str(backup_path or '')
            item['status'] = 'normalized_and_replaced'
        else:
            item['status'] = 'normalized_for_review'
        print(f"OK {name}: {item['status']}")
    except Exception as exc:
        item['status'] = 'error'
        item['error'] = str(exc)
        print(f'ERROR {name}: {exc}')

    results.append(item)

results_df = pd.DataFrame(results)
display(results_df.head(30))
display(results_df['status'].value_counts(dropna=False).reset_index(name='count').rename(columns={'index': 'status'}))

## 4. Exportar resultados

In [ ]:
summary_df = pd.DataFrame([
    {'metric': 'run_timestamp', 'value': run_timestamp},
    {'metric': 'load_results_csv', 'value': str(LOAD_RESULTS_CSV)},
    {'metric': 'footprints_feature_class', 'value': PATH_FC_FOOTPRINTS},
    {'metric': 'rows_to_process', 'value': len(work_df)},
    {'metric': 'replace_originals', 'value': REPLACE_ORIGINALS},
    {'metric': 'create_backup_before_replace', 'value': CREATE_BACKUP_BEFORE_REPLACE},
])

for status, count in results_df['status'].value_counts(dropna=False).items():
    summary_df.loc[len(summary_df)] = {'metric': f'status_{status}', 'value': int(count)}

summary_csv = OUTPUT_DIR / '00_summary.csv'
results_csv = OUTPUT_DIR / '01_normalizacion_rasterio_resultados.csv'
errors_csv = OUTPUT_DIR / '02_errors_review.csv'

summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')
results_df.to_csv(results_csv, index=False, encoding='utf-8-sig')
results_df[results_df['status'].eq('error')].to_csv(errors_csv, index=False, encoding='utf-8-sig')

display(summary_df)
print('Outputs exportados en:', OUTPUT_DIR)
print('CSV resultados:', results_csv)
print('CSV errores:', errors_csv)